In [ ]:
# =====================================================
# 1. 라이브러리
# =====================================================

from pathlib import Path
import re
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from collections import Counter
from wordcloud import WordCloud

In [ ]:
# =====================================================
# 2. 데이터 불러오기
# =====================================================

OUTPUT_DIR = Path(r"C:\Users\soyou\OneDrive\바탕 화면\textdata\arsenal_pipeline_outputs")
INPUT_FILE = OUTPUT_DIR / "arsenal_youtube_cafe_final_labeled.xlsx"

df = pd.read_excel(INPUT_FILE)

print(df.shape)
print(df.columns.tolist())

# 기존 워드클라우드 코드처럼 카페 데이터만 사용
cafe_df = df[df["source"] == "cafe"].copy()

print("\n카페 데이터:", cafe_df.shape)
print("\n38R 전/후 분포:")
print(cafe_df["period"].value_counts(dropna=False))

display(cafe_df[["period", "title", "content_type", "text_clean", "gpt_sentiment"]].head())

# 워드클라우드용 텍스트만 별도 정제
def clean_for_wordcloud(text):
    text = str(text)

    remove_phrases = [
        "이 악성 댓글을 감지합니다",
        "새로운 댓글이 추가될 때 알림을 보내드립니다",
        "댓글 알림 설정이 해제되었습니다",
        "등록순", "최신순", "새로고침", "클린봇", "댓글알림",
        "답글쓰기", "BEST 댓글", "BEST",
    ]

    for p in remove_phrases:
        text = text.replace(p, " ")

    text = re.sub(r"\d{4}\.\d{1,2}\.\d{1,2}\.?\s*\d{1,2}:\d{2}", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


# 일단 카페 댓글만 워드클라우드에 사용
cafe_df = df[
    (df["source"] == "cafe") &
    (df["content_type"] == "cafe_comment")
].copy()

cafe_df["text_wordcloud"] = cafe_df["text_clean"].apply(clean_for_wordcloud)

print("워드클라우드용 카페 댓글:", cafe_df.shape)
display(cafe_df[["period", "text_clean", "text_wordcloud"]].head(20))

In [ ]:
# =====================================================
# 3. 한글 폰트 경로 찾기
# =====================================================

def get_korean_font_path():
    font_candidates = [
        "Malgun Gothic",
        "AppleGothic",
        "NanumGothic",
        "Noto Sans CJK KR"
    ]

    for font in fm.fontManager.ttflist:
        if font.name in font_candidates:
            plt.rcParams["font.family"] = font.name
            plt.rcParams["axes.unicode_minus"] = False
            print(f"WordCloud 한글 폰트 설정: {font.name}")
            return font.fname

    print("한글 폰트를 찾지 못했습니다. font_path를 직접 지정해야 할 수 있습니다.")
    return None

font_path = get_korean_font_path()

In [ ]:
# =====================================================
# 4. 텍스트 전처리 / 토큰화
# =====================================================

stopwords = set([
    # =====================================================
    # 1. 일반 감탄/대화체/의미 약한 표현
    # =====================================================
    "진짜", "너무", "정말", "그냥", "뭔가", "약간", "조금", "정도", "완전", "계속",
    "오늘", "지금", "다시", "이번", "이제", "이후", "당시", "처음", "결국", "동안",
    "이거", "저거", "그거", "이런", "저런", "그런", "여기", "저기", "거기", "다음",
    "사람", "자신", "본인", "모두", "부분", "전체", "이유", "의미", "느낌",
    "마음", "생각", "이야기", "말씀", "말", "하나", "모습", "수준", "이상",
    "ㅋㅋ", "ㅎㅎ", "ㅠㅠ", "ㅜㅜ", "개추", "의외", "덕분", "의문", "사실", 

    # =====================================================
    # 2. 카페/네이버/UI/관리 관련 노이즈
    # =====================================================
    "아스날리쉬", "카페", "네이버", "본문", "댓글", "게시글", "게시", "글", "작성",
    "작성자", "조회", "추천", "공지", "규정", "운영", "관리", "관리자", "진행자",
    "알림", "설정", "해제", "클린봇", "신고", "공유", "링크", "출처", "사진",
    "이미지", "영상", "페이지", "화면", "확인", "선택", "버튼", "메뉴", "목록",
    "구독", "멤버", "회원", "방문", "인기", "베스트", "best",  "프로필",
    "내용", "정보", "질문", "답변", "기록", "검색", "번역", "인쇄", "이름",
    "자격", "단계", "준비", "예정", "상태", "경우", "관련", "사용", "방식",

    # =====================================================
    # 3. 너무 일반적인 축구 단어
    # =====================================================
    "축구", "경기", "시즌", "리그", "프리미어리그", "프리미어", "라운드",
    "전반", "후반", "경기장", "라인업", "선발", "벤치", "주전", "후보",
    "선수", "감독", "팀", "구단", "팬", "응원",
    "골", "득점", "실점", "승리", "패배", "무승부", "결과",
    "수비", "공격", "전술", "교체", "출전", "부상", "복귀",

    # =====================================================
    # Kiwi가 선수/팀명을 잘못 쪼갠 토큰 & 이상용어 추가 제거
    # =====================================================
    "아스", "테타", "세트", "피스", "도련님", "아르",

    
    # =====================================================
    # 4. 아스날 자체 지칭
    # =====================================================
    "아스날", "아스널", "구너", "거너스", "arsenal", "클럽",

    # =====================================================
    # 5. 타 팀명/대회명/지역명 노이즈
    # =====================================================
    "토트넘", "스퍼스", "맨유", "맨시티", "리버풀", "첼시", "뉴캐슬",
    "브라이튼", "빌라", "에버튼", "풀럼", "본머스", "웨스트햄",
    "레알", "바르샤", "바르셀로나", "뮌헨", "psg",
    "잉글랜드", "런던", "프랑스", "스페인", "독일",

    # =====================================================
    # 6. 너무 넓은 평가/상황 단어
    # =====================================================
    "좋음", "최고", "중요", "행복", "기쁨", "기대", "감사", "축하",
    "상상", "역할", "자리", "시간", "시작", "파티", "가족", "분위기",
    "문제", "상황", "가능", "필요", "개인", "특별", "세계", "역사",
    "에너지", "노력", "고생", "확정", "이해", "자체", "증명", "인정", 
])

extra_noise_patterns = [
    "아스날리쉬", "카페", "규정", "알림", "설정", "해제", "클린봇",
    "게시", "작성", "본문", "댓글", "진행자", "관리", "운영",
    "네이버", "회원", "멤버", "구독", "신고", "공유", "링크",
    "페이지", "화면", "메뉴", "목록", "버튼", "이미지", "사진"
]


def remove_cafe_comment_ui_lines(text):
    lines = [x.strip() for x in str(text).splitlines() if x.strip()]
    cleaned = []

    date_pat = re.compile(r"\d{4}\.\d{1,2}\.\d{1,2}\.?\s*\d{1,2}:\d{2}")

    skip_next_nickname = False

    for line in lines:
        if date_pat.search(line):
            continue

        if line == "답글쓰기":
            skip_next_nickname = True
            continue

        if skip_next_nickname and re.fullmatch(r"\d+", line):
            continue

        if skip_next_nickname:
            skip_next_nickname = False
            continue

        if line in [
            "등록순", "최신순", "새로고침", "클린봇",
            "댓글알림", "BEST", "BEST 댓글"
        ]:
            continue

        cleaned.append(line)

    return " ".join(cleaned)


def basic_clean_text(text):
    text = remove_cafe_comment_ui_lines(text)

    ui_phrases = [
        "이 악성 댓글을 감지합니다",
        "새로운 댓글이 추가될 때 알림을 보내드립니다",
        "댓글 알림 설정이 해제되었습니다",
        "댓글알림",
        "클린봇",
        "등록순",
        "최신순",
        "새로고침",
        "답글쓰기",
        "BEST 댓글",
        "BEST",
    ]

    for phrase in ui_phrases:
        text = text.replace(phrase, " ")

    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\d{4}\.\d{1,2}\.\d{1,2}\.?\s*\d{1,2}:\d{2}", " ", text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    USE_KIWI = True
    print("Kiwi 토큰화 사용")
except Exception:
    USE_KIWI = False
    print("Kiwi 없음: 기본 정규식 토큰화 사용")


def is_valid_token(word):
    word = str(word).strip().lower()

    if len(word) < 2:
        return False

    if word in stopwords:
        return False

    if any(noise in word for noise in extra_noise_patterns):
        return False

    if re.fullmatch(r"\d+", word):
        return False

    # 숫자가 섞인 닉네임성 토큰 제거
    if re.search(r"\d", word):
        return False

    if len(word) >= 15:
        return False

    return True


def tokenize_korean(text):
    text = basic_clean_text(text)

    if USE_KIWI:
        tokens = []

        for token in kiwi.tokenize(text):
            if token.tag in ["NNG", "NNP", "SL"]:
                word = token.form.strip().lower()

                if is_valid_token(word):
                    tokens.append(word)

        return tokens

    tokens = re.findall(r"[가-힣a-zA-Z]{2,}", text)

    tokens = [
        t.lower()
        for t in tokens
        if is_valid_token(t)
    ]

    return tokens

In [ ]:
# =====================================================
# 5. 38R 이전 / 이후 텍스트 토큰 만들기
# =====================================================

before_df = cafe_df[cafe_df["period"] == "before_38R"].copy()
after_df = cafe_df[cafe_df["period"] == "after_38R"].copy()

print("38R 이전 데이터:", before_df.shape)
print("38R 이후 데이터:", after_df.shape)

def make_tokens_from_df(data):
    all_tokens = []

    for text in data["text_wordcloud"].dropna():
        all_tokens.extend(tokenize_korean(text))

    return all_tokens

before_tokens = make_tokens_from_df(before_df)
after_tokens = make_tokens_from_df(after_df)

before_counter = Counter(before_tokens)
after_counter = Counter(after_tokens)

print("38R 이전 토큰 수:", len(before_tokens))
print("38R 이후 토큰 수:", len(after_tokens))

print("\n38R 이전 상위 단어")
print(before_counter.most_common(20))

print("\n38R 이후 상위 단어")
print(after_counter.most_common(20))

In [ ]:
# =====================================================
# 6. 워드클라우드 생성 함수
# =====================================================

def plot_wordcloud(counter, title, filename):
    if not counter:
        print("단어가 없어서 워드클라우드를 만들 수 없습니다:", title)
        return

    wc = WordCloud(
        font_path=font_path,
        width=1200,
        height=700,
        background_color="white",
        max_words=150,
        prefer_horizontal=0.9,
        random_state=42
    ).generate_from_frequencies(counter)

    save_path = OUTPUT_DIR / filename

    plt.figure(figsize=(12, 7))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=18, fontweight="bold", pad=18)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("저장 완료:", save_path)

In [ ]:
# =====================================================
# 7. 38R 이전 워드클라우드
# =====================================================

plot_wordcloud(
    before_counter,
    "38R 이전 아스날 카페 워드클라우드",
    "wordcloud_cafe_before_38R.png"
)

In [ ]:
# =====================================================
# 8. 38R 이후 워드클라우드
# =====================================================

plot_wordcloud(
    after_counter,
    "38R 이후 아스날 카페 워드클라우드",
    "wordcloud_cafe_after_38R.png"
)

In [ ]:
# =====================================================
# 9. 상위 단어 막대그래프 함수
# =====================================================

def plot_top_words(counter, title, filename, top_n=20):
    if not counter:
        print("단어가 없어서 그래프를 만들 수 없습니다:", title)
        return

    common = counter.most_common(top_n)
    words = [x[0] for x in common]
    counts = [x[1] for x in common]

    save_path = OUTPUT_DIR / filename

    plt.figure(figsize=(9, 7))
    plt.barh(words[::-1], counts[::-1])
    plt.title(title, fontsize=16, fontweight="bold")
    plt.xlabel("빈도")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("저장 완료:", save_path)

In [ ]:
# =====================================================
# 10. 38R 이전 상위 단어
# =====================================================

plot_top_words(
    before_counter,
    "38R 이전 아스날 카페 상위 단어",
    "top_words_cafe_before_38R.png",
    top_n=20
)

In [ ]:
# =====================================================
# 11. 38R 이후 상위 단어
# =====================================================

plot_top_words(
    after_counter,
    "38R 이후 아스날 카페 상위 단어",
    "top_words_cafe_after_38R.png",
    top_n=20
)